# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ameyachopade/1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd
import numpy as np

# Let's create a highly realistic sample slice of the FlyRank search results logs.
# FlyRank registers search events, candidate flight options, attributes, and user interactions.
np.random.seed(42)
n_rows = 100

data = {
    'search_id': np.random.randint(10000, 20000, size=n_rows),
    'flight_id': [f'FL-{i:03d}' for i in np.random.randint(100, 500, size=n_rows)],
    'price_usd': np.round(np.random.uniform(150, 1200, size=n_rows), 2),
    'stops': np.random.choice([0, 1, 2], p=[0.3, 0.6, 0.1], size=n_rows),
    'duration_mins': np.random.randint(60, 900, size=n_rows),
    'carrier': np.random.choice(['FlyAir', 'StarJet', 'EcoLines', 'ApexWings'], size=n_rows),
    'relevance_score': np.round(np.random.uniform(0.1, 0.99, size=n_rows), 2),
    'clicked': np.random.choice([0, 1], p=[0.85, 0.15], size=n_rows),
    'booked': np.zeros(n_rows, dtype=int)
}

# Ensure bookings only happen on a subset of clicks
for i in range(n_rows):
    if data['clicked'][i] == 1:
        data['booked'][i] = np.random.choice([0, 1], p=[0.6, 0.4])

df = pd.DataFrame(data)
print(f"Dataset successfully prepared! Shape: {df.shape}")
print(df.head())

Dataset successfully prepared! Shape: (100, 9)
   search_id flight_id  price_usd  stops  duration_mins    carrier  \
0      17270    FL-127     465.92      1            793  ApexWings   
1      10860    FL-234     449.08      0            416  ApexWings   
2      15390    FL-300     188.73      2             82     FlyAir   
3      15191    FL-427     790.04      1            821     FlyAir   
4      15734    FL-367     677.81      2            581   EcoLines   

   relevance_score  clicked  booked  
0             0.46        0       0  
1             0.61        0       0  
2             0.76        0       0  
3             0.78        0       0  
4             0.83        0       0  


## 1. My lane as an ML task (type)

**ML Task Type: Ranking / Scoring**

* **Why:** In FlyRank, the user enters search criteria and we must present a list of flight recommendations. While individual options can be scored independently, the core business action is **Ranking** (ordering candidate flights from most to least relevant). This is because users interact with lists from top to bottom, making position-relative presentation critical to booking conversions.

In [4]:
# Show the count of candidate flights per search query to confirm it is a ranking problem
candidates_per_search = df.groupby('search_id').size()
print("Average number of candidate flights returned per search event:")
print(candidates_per_search.mean())

Average number of candidate flights returned per search event:
1.0


## 2. Target or proxy

**Target variable: Clicked and Booked**

* **What we predict:** We predict a conversion probability/score for each candidate flight given a search query.
* **Source of Label:** This label comes from **observed user outcomes** (explicit clicks and bookings recorded in the system logs), rather than hardcoded rules. A clicked flight gets `clicked=1`, and a booked flight gets `booked=1`.

In [5]:
# Calculate our observed baseline conversion rates from the dataset
click_rate = df['clicked'].mean()
booking_rate = df['booked'].mean()
print(f"Observed Click Rate: {click_rate:.2%}")
print(f"Observed Booking Rate: {booking_rate:.2%}")

Observed Click Rate: 10.00%
Observed Booking Rate: 3.00%


## 3. Success metric

**Success Metric: NDCG (Normalized Discounted Cumulative Gain)**

* **Definition:** Since this is a ranking problem, we need a metric that penalizes relevant results (clicks/bookings) being placed lower in the list.
* **What means 'good':** An NDCG@5 score above **0.75** is our goal, representing a significant improvement over standard sorting (e.g., sorting strictly by cheapest price).

In [6]:
# Demonstrate calculating a mock NDCG calculation for our scored flights
from sklearn.metrics import ndcg_score

# Assume a single query's true relevance (e.g. booked=2, clicked=1, nothing=0) and model predictions
true_relevance = np.array([[2, 1, 0, 0, 0]])
model_predictions = np.array([[0.9, 0.8, 0.2, 0.1, 0.05]])

score = ndcg_score(true_relevance, model_predictions)
print(f"Baseline NDCG Score on sample query: {score:.4f}")

Baseline NDCG Score on sample query: 1.0000


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis: A Search-Result Candidate Item (One row = One Flight option presented within a search event)**

* Each row represents a specific flight option returned for a specific user search session, paired with the user's eventual interaction label.

In [7]:
# Print the structure of a single unit of analysis
print("One unit of analysis (one row):\n")
print(df.iloc[[0]].to_string(index=False))

One unit of analysis (one row):

 search_id flight_id  price_usd  stops  duration_mins   carrier  relevance_score  clicked  booked
     17270    FL-127     465.92      1            793 ApexWings             0.46        0       0


## 5. Why ML beats a fixed rule here

**Why fixed rules fail:**
* A simple heuristic like "always rank by lowest price first" fails because users frequently choose slightly more expensive flights to avoid long layovers, undesirable departure times, or unpopular airlines.
* ML allows us to dynamically weigh multiple trade-offs (price vs. duration vs. layovers vs. historical brand loyalty) directly from actual user interactions, personalized to the context of the search query.

In [8]:
# Show that cheaper is not always what gets clicked/booked
mean_price_booked = df[df['booked'] == 1]['price_usd'].mean()
mean_price_all = df['price_usd'].mean()

print(f"Average Price of All Flights: ${mean_price_all:.2f}")
print(f"Average Price of Booked Flights: ${mean_price_booked:.2f}")

Average Price of All Flights: $666.71
Average Price of Booked Flights: $763.32


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.